# Detailed comparisons between MeshFEM and TinyAD
Test different variants of MeshFEM against TinyAD and a hybrid of MeshFEM eval + TinyAD Newton.

In [ ]:
import psutil
nthreads = psutil.cpu_count(logical=False)

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = str(nthreads)

import sys, os
sys.path.append('../')
import MeshFEM
import mesh, mesh_energy, py_newton_optimizer, viewer
import parametrization, benchmark, flip_avoiding_step_length
import energy

import numpy as np
import time, copy
import param_utils
import helper_funcs
from helper_funcs import HessianStats
import igl

import psutil
import parallelism
parallelism.set_max_num_tbb_threads(nthreads)

In [ ]:
import tinyad_parametrization, dirichlet_demo

In [ ]:
clamp_eps = 1e-9

In [ ]:
mesh_file_path = '../../models/Superman_cut2.off'
mesh_file_path = '../../models/Lucy_3cuts.msh.xz'
m = helper_funcs.read_mesh(mesh_file_path)

In [ ]:
uv = mesh_energy.NodalVars(m, 2)
bdry_uv = helper_funcs.getBDdataOnNormalizedCircle(m)
# Tutte Initialization
uv_init = parametrization.harmonic(m, bdry_uv)
flip_list = parametrization.getFlips(m, uv_init)
if len(flip_list) > 0:  uv_init = parametrization.harmonic(m, bdry_uv, True)
uv.setVars(uv_init.ravel())

In [ ]:
params = [
    mesh_energy.Parametrization(m, uv, energy.SymmetricDirichletDerivativeFree(2)), # Standard F-autodiff
    dirichlet_demo.param_symdirichlet_element_ad(m, uv), # x-autodiff of custom Symmetric Dirichlet formulas
    dirichlet_demo.param_symdirichlet_element_tad_compare(m, uv), # x-autodiff using the same (inefficient) formulas from the TinyAD demo
]
problems = [py_newton_optimizer.NewtonMultiobjectiveProblem(uv, [p]) for p in params]

# Configure settings to match TinyAD
for p in params:
    p.useXBasedProjection = True
    p.elementHessianShift = 0.0
    p.xBasedProjectionClampEps = clamp_eps
for p in problems: p.hessianShift = 0.0

# Hessian comparisons

In [ ]:
f, g, H_proj = tinyad_parametrization.symmdsParamTinyADEvalFGH(m, uv_init.ravel(), project=True, proj_eps = clamp_eps)

In [ ]:
H_m = [p.hessian(projectionMask = True).toSciPy(upperTriangleOnly=False) for p in params]

In [ ]:
H_diffs = [H - H_proj for H in H_m]

In [ ]:
[np.linalg.norm(Hd.data) / np.linalg.norm(H_proj.data) for Hd in H_diffs]

# Optimizations

In [ ]:
def run_optimization(p):
    opt = p.optimizer()
    opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()
    opt.options.backtrack_shrink_factor = 0.8
    opt.options.nbacktrack_iter = 64
    opt.options.niter = 50
    
    uv.setVars(uv_init.ravel())
    benchmark.reset()
    cr = opt.optimize()
    benchmark.report()
    return cr

In [ ]:
# Run MeshFEM variants
crs = [run_optimization(p) for p in problems]

In [ ]:
benchmark.reset()
tad_gnorm = tinyad_parametrization.symmdsParamTinyAD(m, uv_init, 50, 1e-7, proj_eps=clamp_eps)[2]
benchmark.report()

In [ ]:
benchmark.reset()
hybrid_gnorm = tinyad_parametrization.paramTADMeshFEMHybrid(m, uv_init, 50, 1e-7, proj_eps=clamp_eps)[2]
benchmark.report()

In [ ]:
from matplotlib import pyplot as plt
for i, cr in enumerate(crs):
    plt.semilogy(cr.freeGradientNorm, label=f'MeshFEM{i + 1}', lw=5-2 * i)
plt.semilogy(tad_gnorm, label='TAD')
plt.semilogy(hybrid_gnorm, label='MeshFEM+TAD Hybrid')
plt.legend()